# Spotify Charts Analysis

Исследование музыкальных чартов: шесть гипотез, визуализации и статистические проверки.

- [Описание проекта и установка](README.md)
- [Входные данные и ограничения](docs/DATA.md)

Для повторного выполнения нужен `Top_spotify_songs.csv` рядом с ноутбуком. Сохранённые выводы относятся к прежнему запуску.


Курсовая работа "Аналитика Spotify"

 **Данные о столбцах**  
***spotify_id*** - Уникальный идентификатор песни в базе данных Spotify.  
***name*** - Название песни.  
***artists*** - Имя (имена) исполнителя (исполнителей), связанного с песней.  
***daily_rank*** - Ежедневное попадание песни в список 50 лучших песен за день.  
***daily_movement*** - Изменение рейтинга по сравнению с предыдущим днем.  
***weekly_movement*** - Изменение рейтинга по сравнению с предыдущей неделей.  
***country*** - Код ISO страны, в которой находится плейлист «Топ-50». Если значение равно NULL, то плейлист называется «Глобальный Топ-50».  
***snapshot_date*** - Дата, когда данные были собраны с помощью Spotify API.  
***popularity*** - Показатель текущей популярности песни на Spotify.  
***is_explicit*** - Указывает, содержит ли песня явный текст.    
***duration_ms*** - Продолжительность песни в миллисекундах.  
***album_name*** - Название альбома, к которому принадлежит песня.  
***album_release_date*** - Дата выхода альбома, к которому относится песня.    
***danceability*** - Показатель того, насколько песня подходит для танцев, на основе различных музыкальных элементов.  
***energy*** - Показатель интенсивности и уровня активности песни.  
***key*** - Ключ к песне.  
***loudness*** - Общая громкость песни в децибелах.  
***mode*** - Указывает, написана ли песня в мажорной или минорной тональности.  
***speechiness*** - Показатель наличия произнесенных слов в песне.  
***acousticness*** - Показатель акустического качества песни.  
***instrumentalness*** - Показатель вероятности того, что песня не содержит вокала.  
***liveness*** - Показатель присутствия живой аудитории на записи.  
***valence*** - Мера музыкального позитива, передаваемого песней.  
***tempo*** - Темп песни в ударах в минуту.  
***time_signature*** - Предполагаемый общий временной интервал песни.




гипотеза 1
Акустические треки со средним темпом (100-130 BPM) имеют более высокую и стабильную популярность по сравнению с очень медленными (<70 BPM) или очень быстрыми (>160 BPM) композициями

гипотеза 2
Акустические ремиксы (содержащие "Remix" в названии) имеют статистически более высокие значения danceability и energy по сравнению с оригинальными акустическими треками, при этом сохраняя высокий уровень acousticnes и valence.

гипотеза 3 
В латиноамериканских чартах средние значения danceability и energy будут статистически выше, а в азиатских — выше acousticness и loudness.

гипотеза 4
Показатель валентности (valence) треков в топ-чартах систематически изменяется в течение недели: в начале недели показатели в сторону грустных песен, в конце недели - наоборот

гипотеза 5
Треки, выпущенные весной/в начале лета (album_release_date с апреля по июль), имеют статистически более высокие показатели energy, danceability и valence по сравнению с треками, выпущенными осенью/зимой (с октября по январь), которые будут более акустичными (acousticness) и меланхоличными (низкая valence).

гипотеза 6
Гипотеза: За последние 5 лет в мировой поп-музыке (глобальный чарт) наметилась тенденция к снижению "веселья" и "энергичности". А именно, с 2018 по 2025 год наблюдается статистически значимое снижение средних показателей valence (позитивность) и energy, при этом растет показатель acousticness.

In [1]:
import pandas as pd
import pycountry

In [2]:
df = pd.read_csv('Top_spotify_songs.csv')

In [3]:
df.columns

Index(['spotify_id', 'name', 'artists', 'daily_rank', 'daily_movement',
       'weekly_movement', 'country', 'snapshot_date', 'popularity',
       'is_explicit', 'duration_ms', 'album_name', 'album_release_date',
       'danceability', 'energy', 'key', 'loudness', 'mode', 'speechiness',
       'acousticness', 'instrumentalness', 'liveness', 'valence', 'tempo',
       'time_signature'],
      dtype='object')

In [4]:
df

,spotify_id,name,artists,daily_rank,daily_movement,weekly_movement,country,snapshot_date,popularity,is_explicit,...,key,loudness,mode,speechiness,acousticness,instrumentalness,liveness,valence,tempo,time_signature
0,2plbrEY59IikOBgBGLjaoe,Die With A Smile,"Lady Gaga, Bruno Mars",1,1,0,NaN,2025-02-17,98,False,...,6,-7.777,0,0.0304,0.3080,0.000000,0.122,0.535,157.969,3
1,2CGNAOSuO1MEFCbBRgUzjd,luther (with sza),"Kendrick Lamar, SZA",2,1,4,NaN,2025-02-17,90,False,...,2,-7.546,1,0.1250,0.2510,0.000000,0.248,0.576,138.008,4
2,6AI3ezQ4o3HUoP6Dhudph3,Not Like Us,Kendrick Lamar,3,-2,8,NaN,2025-02-17,92,True,...,1,-7.001,1,0.0776,0.0107,0.000000,0.141,0.214,101.061,4
3,4wJ5Qq0jBN4ajy7ouZIV1c,APT.,"ROSÉ, Bruno Mars",4,0,-2,NaN,2025-02-17,89,False,...,0,-4.477,0,0.2600,0.0283,0.000000,0.355,0.939,149.027,4
4,6dOtVTDdiauQNBQEDOtlAB,BIRDS OF A FEATHER,Billie Eilish,5,1,-2,NaN,2025-02-17,96,False,...,2,-10.171,1,0.0358,0.2000,0.060800,0.117,0.438,104.978,4
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1728377,0AYt6NMyyLd0rLuvr0UkMH,Slime You Out (feat. SZA),"Drake, SZA",46,4,0,AE,2023-10-18,84,True,...,5,-9.243,0,0.0502,0.5080,0.000000,0.259,0.105,88.880,3
1728378,2Gk6fi0dqt91NKvlzGsmm7,SAY MY GRACE (feat. Travis Scott),"Offset, Travis Scott",47,3,0,AE,2023-10-18,80,True,...,10,-5.060,1,0.0452,0.0585,0.000000,0.132,0.476,121.879,4
1728379,26b3oVLrRUaaybJulow9kz,People,Libianca,48,2,0,AE,2023-10-18,88,False,...,10,-7.621,0,0.0678,0.5510,0.000013,0.102,0.693,124.357,5
1728380,5ydjxBSUIDn26MFzU3asP4,Rainy Days,V,49,1,0,AE,2023-10-18,88,False,...,9,-8.016,0,0.0875,0.7390,0.000000,0.148,0.282,74.828,4


In [5]:
df = df.drop_duplicates()
df = df.dropna()


In [6]:
df

,spotify_id,name,artists,daily_rank,daily_movement,weekly_movement,country,snapshot_date,popularity,is_explicit,...,key,loudness,mode,speechiness,acousticness,instrumentalness,liveness,valence,tempo,time_signature
50,2CGNAOSuO1MEFCbBRgUzjd,luther (with sza),"Kendrick Lamar, SZA",1,3,10,ZA,2025-02-17,90,False,...,2,-7.546,1,0.1250,0.2510,0.000000,0.2480,0.576,138.008,4
51,021iZJ1jjc6cZZ4sMXvcW9,Romeo & Juliet,"Naledi Aphiwe, Mawelele",2,3,2,ZA,2025-02-17,58,False,...,2,-9.515,1,0.0475,0.2410,0.017900,0.0593,0.691,125.989,3
52,24fRfn3FOM4xk02aE4wief,Vuka (feat. Thukuthela),"Oscar Mbo, Jazzworx, Thukuthela",3,-1,-2,ZA,2025-02-17,62,False,...,4,-10.965,0,0.0466,0.1750,0.198000,0.1050,0.524,118.006,4
53,2LeOHKRLm1Bz7eeNVinUYs,Esandleni,Nontokozo Mkhize,4,19,-2,ZA,2025-02-17,52,False,...,4,-5.893,1,0.0395,0.0597,0.000450,0.3350,0.495,99.768,4
54,4QdFYz1hjkDVDTqAKmIdOx,"Ngibolekeni (feat. Seun1401, LeeMcKrazy, Blxck...","DJ Maphorisa, Xduppy, Scotts Maphuma, Seun1401...",5,-4,-2,ZA,2025-02-17,59,False,...,1,-8.950,1,0.0546,0.0126,0.000281,0.0609,0.746,112.012,4
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1728377,0AYt6NMyyLd0rLuvr0UkMH,Slime You Out (feat. SZA),"Drake, SZA",46,4,0,AE,2023-10-18,84,True,...,5,-9.243,0,0.0502,0.5080,0.000000,0.2590,0.105,88.880,3
1728378,2Gk6fi0dqt91NKvlzGsmm7,SAY MY GRACE (feat. Travis Scott),"Offset, Travis Scott",47,3,0,AE,2023-10-18,80,True,...,10,-5.060,1,0.0452,0.0585,0.000000,0.1320,0.476,121.879,4
1728379,26b3oVLrRUaaybJulow9kz,People,Libianca,48,2,0,AE,2023-10-18,88,False,...,10,-7.621,0,0.0678,0.5510,0.000013,0.1020,0.693,124.357,5
1728380,5ydjxBSUIDn26MFzU3asP4,Rainy Days,V,49,1,0,AE,2023-10-18,88,False,...,9,-8.016,0,0.0875,0.7390,0.000000,0.1480,0.282,74.828,4


In [7]:
df.iloc[0]

spotify_id            2CGNAOSuO1MEFCbBRgUzjd
name                       luther (with sza)
artists                  Kendrick Lamar, SZA
daily_rank                                 1
daily_movement                             3
weekly_movement                           10
country                                   ZA
snapshot_date                     2025-02-17
popularity                                90
is_explicit                            False
duration_ms                           177598
album_name                               GNX
album_release_date                2024-11-21
danceability                           0.707
energy                                 0.575
key                                        2
loudness                              -7.546
mode                                       1
speechiness                            0.125
acousticness                           0.251
instrumentalness                         0.0
liveness                               0.248
valence   

In [8]:
df.info()

<class 'pandas.core.frame.DataFrame'>
Index: 1703961 entries, 50 to 1728381
Data columns (total 25 columns):
 #   Column              Dtype  
---  ------              -----  
 0   spotify_id          object 
 1   name                object 
 2   artists             object 
 3   daily_rank          int64  
 4   daily_movement      int64  
 5   weekly_movement     int64  
 6   country             object 
 7   snapshot_date       object 
 8   popularity          int64  
 9   is_explicit         bool   
 10  duration_ms         int64  
 11  album_name          object 
 12  album_release_date  object 
 13  danceability        float64
 14  energy              float64
 15  key                 int64  
 16  loudness            float64
 17  mode                int64  
 18  speechiness         float64
 19  acousticness        float64
 20  instrumentalness    float64
 21  liveness            float64
 22  valence             float64
 23  tempo               float64
 24  time_signature      int64  


Гипотеза 1

In [9]:
def category_tempo(tempo):
    if tempo < 70:
        return 'Медленные'
    elif 70 <= tempo <= 130:
        return 'Средние'
    else:
        return 'Быстрые'

df['tempo_category'] = df['tempo'].apply(category_tempo)

In [10]:
tempo_stats = df.groupby('tempo_category')['popularity'].agg(['mean', 'median', 'std', 'count'])

In [11]:
tempo_stats

,mean,median,std,count
tempo_category,,,,
Быстрые,75.466788,79.0,16.157816,593541
Медленные,82.362193,88.0,14.362518,14738
Средние,76.085371,80.0,15.494424,1095682


In [12]:
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import plotly.colors as colors

color_sequence = ['#FF6B6B', '#4ECDC4', '#45B7D1']

fig = make_subplots(
    rows=3, cols=1,
    subplot_titles=(
        'Распределение популярности по группам темпа',
        'Плотность распределения популярности', 
        'Средняя популярность по диапазонам темпа'
    ),
    specs=[[{"type": "box"}],
           [{"type": "violin"}],
           [{"type": "bar"}]],
    vertical_spacing=0.08,
    horizontal_spacing=0.08
)

for i, category in enumerate(df['tempo_category'].unique()):
    category_data = df[df['tempo_category'] == category]['popularity']
    fig.add_trace(
        go.Box(
            y=category_data,
            name=category,
            marker_color=color_sequence[i],
            boxpoints=False,
            line_width=2,
            whiskerwidth=0.2
        ),
        row=1, col=1
    )

for i, category in enumerate(df['tempo_category'].unique()):
    category_data = df[df['tempo_category'] == category]['popularity']
    fig.add_trace(
        go.Violin(
            y=category_data,
            name=category,
            marker_color=color_sequence[i],
            box_visible=True,
            meanline_visible=True,
            line_width=1,
            points=False
        ),
        row=2, col=1
    )

tempo_bins = [0, 60, 80, 100, 120, 140, 160, 200, 300]
tempo_labels = ['<60', '60-80', '80-100', '100-120', '120-140', '140-160', '160-200', '>200']
df['tempo_range'] = pd.cut(df['tempo'], bins=tempo_bins, labels=tempo_labels)

popularity_by_tempo = df.groupby('tempo_range', observed=True)['popularity'].mean().reset_index()

fig.add_trace(
    go.Bar(
        x=popularity_by_tempo['tempo_range'],
        y=popularity_by_tempo['popularity'],
        marker_color=px.colors.sequential.Viridis,
        hovertemplate='<b>Диапазон:</b> %{x} BPM<br><b>Популярность:</b> %{y:.2f}<extra></extra>',
        name='Средняя популярность'
    ),
    row=3, col=1
)

fig.update_layout(
    height=1200, 
    title_text="Анализ влияния темпа на популярность акустических треков",
    title_x=0.5,
    title_font_size=20,
    showlegend=True,
    legend=dict(
        orientation="h",
        yanchor="bottom",
        y=1.02,
        xanchor="center",
        x=0.5
    ),
    font=dict(family="Arial, sans-serif", size=12),
    plot_bgcolor='rgba(248,248,248,0.8)',
    paper_bgcolor='white'
)

fig.update_xaxes(title_text="Группы темпа", row=1, col=1)
fig.update_xaxes(title_text="Группы темпа", row=2, col=1)
fig.update_xaxes(title_text="Диапазон темпа (BPM)", row=3, col=1)

fig.update_yaxes(title_text="Популярность", row=1, col=1)
fig.update_yaxes(title_text="Популярность", row=2, col=1)
fig.update_yaxes(title_text="Средняя популярность", row=3, col=1)

fig.show()

In [13]:
from scipy.stats import f_oneway

slow_tempo = df[df['tempo_category'] == 'Медленные']['popularity']
medium_tempo = df[df['tempo_category'] == 'Средние']['popularity']
fast_tempo = df[df['tempo_category'] == 'Быстрые']['popularity']

f_stat, p_value = f_oneway(slow_tempo, medium_tempo, fast_tempo)
print(f"F-статистика: {f_stat}, p-значение: {p_value}")
print(f"  Значимо: {'Да' if p_value < 0.05 else 'Нет'}")

F-статистика: 1544.9320443797033, p-значение: 0.0
  Значимо: Да


Гипотеза 2

In [14]:
Remix = df[df['name'].str.find('Remix') >= 0]
No_Remix = df[df['name'].str.find('Remix') < 0]

In [15]:
fig = make_subplots(
    rows=1, cols=2,
    subplot_titles=(
        'Сравнение ключевых параметров', 
        'Распределение популярности'
    ),
    specs=[[{"type": "bar"}, {"type": "box"}]],
    horizontal_spacing=0.1
)

comparison_features = ['danceability', 'energy', 'acousticness', 'valence']
remix_means = [Remix[f].mean() for f in comparison_features]
original_means = [No_Remix[f].mean() for f in comparison_features]

fig.add_trace(
    go.Bar(name='Ремиксы', x=comparison_features, y=remix_means,
           marker_color='#e74c3c', text=remix_means, 
           texttemplate='%{text:.3f}', textposition='outside',
           width=0.4),
    row=1, col=1
)

fig.add_trace(
    go.Bar(name='Оригиналы', x=comparison_features, y=original_means,
           marker_color='#3498db', text=original_means,
           texttemplate='%{text:.3f}', textposition='outside',
           width=0.4),
    row=1, col=1
)
fig.add_trace(
    go.Box(y=No_Remix['popularity'], name='Оригиналы',
           marker_color='#3498db', boxpoints=False,
           line_width=2),
    row=1, col=2
)

fig.add_trace(
    go.Box(y=Remix['popularity'], name='Ремиксы',
           marker_color='#e74c3c', boxpoints=False,
           line_width=2),
    row=1, col=2
)

fig.update_layout(
    height=500,
    title_text="Сравнение ремиксов и оригиналов",
    title_x=0.5,
    title_font_size=16,
    showlegend=True,
    font=dict(family="Arial, sans-serif", size=12),
    plot_bgcolor='rgba(248,248,248,0.9)',
    paper_bgcolor='white',
    bargap=0.3,
    bargroupgap=0.1
)

fig.update_xaxes(title_text="Параметры", row=1, col=1)
fig.update_xaxes(title_text="Тип трека", row=1, col=2)

fig.update_yaxes(title_text="Значение параметра", row=1, col=1, range=[0, 1])
fig.update_yaxes(title_text="Популярность", row=1, col=2)

fig.show()


In [16]:
from scipy.stats import ttest_ind
for feature in ['danceability', 'energy', 'acousticness', 'valence']:
    t_stat, p_val = ttest_ind(Remix[feature], No_Remix[feature], equal_var=False)
    
    remix_mean = Remix[feature].mean()
    original_mean = No_Remix[feature].mean()
    
    print(f"\n{feature}:")
    print(f"  Ремиксы:    {remix_mean:.3f}")
    print(f"  Оригиналы:  {original_mean:.3f}")
    print(f"  p-value:    {p_val:.4f}")
    
    if p_val > 0.05:
        print(" Статистическая значимость: p > 0.05 - различия не значимы")
    else:
        print(" Статистическая значимость: p < 0.05 - различия значимы")



danceability:
  Ремиксы:    0.714
  Оригиналы:  0.680
  p-value:    0.0000
 Статистическая значимость: p < 0.05 - различия значимы

energy:
  Ремиксы:    0.697
  Оригиналы:  0.649
  p-value:    0.0000
 Статистическая значимость: p < 0.05 - различия значимы

acousticness:
  Ремиксы:    0.213
  Оригиналы:  0.275
  p-value:    0.0000
 Статистическая значимость: p < 0.05 - различия значимы

valence:
  Ремиксы:    0.544
  Оригиналы:  0.550
  p-value:    0.0012
 Статистическая значимость: p < 0.05 - различия значимы


Гипотеза 3

In [30]:
df1 = df[df['country'] != 'All']
df1 = df1.groupby('country').agg({'energy': 'mean', 'loudness': 'median', 'danceability': 'mean', 'acousticness': 'mean'})
df1['energy'] = df1['energy'].apply(lambda x: round(x, 2))
df1['danceability'] = df1['danceability'].apply(lambda x: round(x, 2))
df1['loudness'] = df1['loudness'].apply(lambda x: round(x, 2))
df1['acousticness'] = df1['acousticness'].apply(lambda x: round(x, 2))
df1 = pd.DataFrame(df1)
df1.reset_index(inplace=True)
df1['country_name'] = df1['country'].apply(lambda x: pycountry.countries.get(alpha_2=x).name)
df1['country'] = df1['country'].apply(lambda x: pycountry.countries.get(alpha_2=x).alpha_3)
df1['country'] = df1['country'].apply(lambda x: str(x.split(',')[0]))

In [31]:
import plotly.express as px
fig = px.choropleth(df1, locations='country', color= 'energy', hover_name='country_name', color_continuous_scale = 'Greens')
fig.update_layout(
    height=650,
    width=1000)

menu = [
    {'buttons': [{'method': 'restyle', 'label': 'Средняя энергичность треков', 'args': [{'z': [df1['energy']], 'color': [df1['energy']]}]},
    {'method': 'restyle', 'label': 'Средняя танцевальнось треков', 'args': [{'z': [df1['danceability']], 'color': [df1['danceability']]}]},
    {'method': 'restyle', 'label': 'Средняя громкость треков', 'args': [{'z': [df1['loudness']], 'color': [df1['loudness']]}]},
    {'method': 'restyle', 'label': 'Средняя акустичность треков', 'args': [{'z': [df1['acousticness']], 'color': [df1['acousticness']]}]}],
    'direction': 'down',
    'showactive': True,
    'x': 0.24,
    'y': 1.05}]

fig.update_layout(updatemenus= menu, height= 650,
    width= 1000 )

fig.show()

In [19]:
latin_countries = ['AR', 'BR', 'CL', 'CO', 'MX', 'PE', 'UY']  
asian_countries = ['JP', 'KR', 'ID', 'TH', 'VN', 'PH', 'MY']  

df_latin = df[df['country'].isin(latin_countries)]
df_asia = df[df['country'].isin(asian_countries)]

In [20]:

audio_features = ['danceability', 'energy', 'acousticness', 'loudness', 
                  'valence', 'tempo', 'speechiness', 'instrumentalness']

In [21]:
corr_latin = df_latin[audio_features].corr()
corr_asia = df_asia[audio_features].corr()

In [ ]:
fig_latin = px.imshow(
    corr_latin,
    text_auto=True,  
    color_continuous_scale='RdBu_r',  
    zmin=-1, zmax=1,  
    title='Корреляция аудио-признаков: Латинская Америка',
    labels=dict(color="Корреляция"),
    width=800, height=700
)


fig_latin.show()

fig_asia = px.imshow(
    corr_asia,
    text_auto=True,
    color_continuous_scale='RdBu_r',
    zmin=-1, zmax=1,
    title='Корреляция аудио-признаков: Азия',
    labels=dict(color="Корреляция"),
    width=800, height=700
)

fig_asia.show()

In [ ]:
from scipy import stats
dance_latin = df_latin['danceability'].dropna()
dance_asia = df_asia['danceability'].dropna()
t_dance, p_dance = stats.ttest_ind(dance_latin, dance_asia, equal_var=False)
print(f"1. DANCEABILITY:")
print(f"   Латинская Америка: среднее = {dance_latin.mean():.3f}, n = {len(dance_latin):,}")
print(f"   Азия: среднее = {dance_asia.mean():.3f}, n = {len(dance_asia):,}")
print(f"   t({len(dance_latin)+len(dance_asia)-2}) = {t_dance:.3f}, p = {p_dance:.6f}")
print(f"   Вывод: {'Латинская Америка > Азия' if dance_latin.mean() > dance_asia.mean() else 'Азия > Латинская Америка'}")
print(f"   Статистически значимо: {'да' if p_dance < 0.05 else 'нет'}")

energy_latin = df_latin['energy'].dropna()
energy_asia = df_asia['energy'].dropna()
t_energy, p_energy = stats.ttest_ind(energy_latin, energy_asia, equal_var=False)
print(f"2. ENERGY:")
print(f"   Латинская Америка: среднее = {energy_latin.mean():.3f}, n = {len(energy_latin):,}")
print(f"   Азия: среднее = {energy_asia.mean():.3f}, n = {len(energy_asia):,}")
print(f"   t({len(energy_latin)+len(energy_asia)-2}) = {t_energy:.3f}, p = {p_energy:.6f}")
print(f"   Вывод: {'Латинская Америка > Азия' if energy_latin.mean() > energy_asia.mean() else 'Азия > Латинская Америка'}")
print(f"   Статистически значимо: {'да' if p_energy < 0.05 else 'нет'}")
print()

acoustic_latin = df_latin['acousticness'].dropna()
acoustic_asia = df_asia['acousticness'].dropna()
t_acoustic, p_acoustic = stats.ttest_ind(acoustic_latin, acoustic_asia, equal_var=False)
print(f"3. ACOUSTICNESS:")
print(f"   Латинская Америка: среднее = {acoustic_latin.mean():.3f}, n = {len(acoustic_latin):,}")
print(f"   Азия: среднее = {acoustic_asia.mean():.3f}, n = {len(acoustic_asia):,}")
print(f"   t({len(acoustic_latin)+len(acoustic_asia)-2}) = {t_acoustic:.3f}, p = {p_acoustic:.6f}")
print(f"   Вывод: {'Латинская Америка > Азия' if acoustic_latin.mean() > acoustic_asia.mean() else 'Азия > Латинская Америка'}")
print(f"   Статистически значимо: {'да' if p_acoustic < 0.05 else 'нет'}")

loud_latin = df_latin['loudness'].dropna()
loud_asia = df_asia['loudness'].dropna()
t_loud, p_loud = stats.ttest_ind(loud_latin, loud_asia, equal_var=False)
print(f"4. LOUDNESS:")
print(f"   Латинская Америка: среднее = {loud_latin.mean():.3f}, n = {len(loud_latin):,}")
print(f"   Азия: среднее = {loud_asia.mean():.3f}, n = {len(loud_asia):,}")
print(f"   t({len(loud_latin)+len(loud_asia)-2}) = {t_loud:.3f}, p = {p_loud:.6f}")
print(f"   Вывод: {'Латинская Америка > Азия' if loud_latin.mean() > loud_asia.mean() else 'Азия > Латинская Америка'}")
print(f"   Статистически значимо: {'да' if p_loud < 0.05 else 'нет'}")

1. DANCEABILITY:
   Латинская Америка: среднее = 0.727, n = 165,789
   Азия: среднее = 0.617, n = 166,156
   t(331943) = 239.949, p = 0.000000
   Вывод: Латинская Америка > Азия
   Статистически значимо: ДА
2. ENERGY:
   Латинская Америка: среднее = 0.691, n = 165,789
   Азия: среднее = 0.613, n = 166,156
   t(331943) = 136.562, p = 0.000000
   Вывод: Латинская Америка > Азия
   Статистически значимо: ДА

3. ACOUSTICNESS:
   Латинская Америка: среднее = 0.265, n = 165,789
   Азия: среднее = 0.337, n = 166,156
   t(331943) = -79.725, p = 0.000000
   Вывод: Азия > Латинская Америка
   Статистически значимо: ДА
4. LOUDNESS:
   Латинская Америка: среднее = -5.685, n = 165,789
   Азия: среднее = -6.711, n = 166,156
   t(331943) = 87.032, p = 0.000000
   Вывод: Латинская Америка > Азия
   Статистически значимо: ДА


Гипотеза 4

In [ ]:
df['snapshot_date'] = pd.to_datetime(df['snapshot_date'])
df['day_of_week'] = df['snapshot_date'].dt.day_name()
df['day_number'] = df['snapshot_date'].dt.dayofweek 

In [ ]:

daily_valence = df.groupby('day_of_week', as_index=False).agg({
    'valence': 'mean',
    'spotify_id': 'count',  
    'danceability': 'mean',
    'energy': 'mean'
}).rename(columns={'spotify_id': 'track_count'})

In [ ]:
daily_valence

,day_of_week,valence,track_count,danceability,energy
0,Friday,0.548072,243821,0.679891,0.648999
1,Monday,0.553159,245614,0.679674,0.650311
2,Saturday,0.547865,243731,0.680930,0.650167
3,Sunday,0.551375,238019,0.681890,0.652349
4,Thursday,0.548242,244245,0.679367,0.648608
5,Tuesday,0.551603,242126,0.678679,0.648418
6,Wednesday,0.549206,246405,0.678911,0.648313


In [ ]:
days_order = ['Monday', 'Tuesday', 'Wednesday', 'Thursday', 'Friday', 'Saturday', 'Sunday']
daily_valence['day_of_week'] = pd.Categorical(daily_valence['day_of_week'], categories=days_order, ordered=True)
daily_valence = daily_valence.sort_values('day_of_week')

In [ ]:
fig = go.Figure()

fig.add_trace(go.Scatter(
    x=daily_valence['day_of_week'],
    y=daily_valence['valence'],
    mode='lines+markers+text',
    name='Валентность',
    line=dict(width=4, color='#FF6B6B'),
    marker=dict(size=12),
    text=[f'{val:.3f}' for val in daily_valence['valence']],
    textposition='top center'
))

fig.update_layout(
    title='Тренд валентности по дням недели',
    xaxis_title='День недели',
    yaxis_title='Средняя валентность',
    yaxis_range=[daily_valence['valence'].min() - 0.1, daily_valence['valence'].max() + 0.1],
    height=500
)

fig.show()

fig = make_subplots(rows=2, cols=1, 
                   subplot_titles=('Танцевальность по дням недели', 'Энергичность по дням недели'))

fig.add_trace(go.Scatter(x=daily_valence['day_of_week'], y=daily_valence['danceability'],
                         mode='lines+markers', name='Danceability', line=dict(color='#4ECDC4')), row=1, col=1)

fig.add_trace(go.Scatter(x=daily_valence['day_of_week'], y=daily_valence['energy'],
                         mode='lines+markers', name='Energy', line=dict(color='#45B7D1')), row=2, col=1)

fig.update_layout(height=600, title_text="Танцевальность и энергичность по дням недели")
fig.show()

corr_dance = daily_valence['valence'].corr(daily_valence['danceability'])
corr_energy = daily_valence['valence'].corr(daily_valence['energy'])

print(f"Корреляция валентности с танцевальностью: {corr_dance:.3f}")
print(f"Корреляция валентности с энергичностью: {corr_energy:.3f}")

Корреляция валентности с танцевальностью: -0.027
Корреляция валентности с энергичностью: 0.372


In [ ]:
from scipy import stats
weekend_days = ['Friday', 'Saturday', 'Sunday']
weekday_days = ['Monday', 'Tuesday', 'Wednesday', 'Thursday']
weekend_valence = daily_valence[daily_valence['day_of_week'].isin(weekend_days)]['valence'].mean()
weekday_valence = daily_valence[daily_valence['day_of_week'].isin(weekday_days)]['valence'].mean()

difference = weekend_valence - weekday_valence
percent_difference = (difference / weekday_valence) * 100

print(f"Средняя валентность в будни:  {weekday_valence:.3f}")
print(f"Средняя валентность в выходные: {weekend_valence:.3f}")
print(f"Разница: {difference:+.3f} ({percent_difference:+.1f}%)")

weekend_tracks = df[df['day_of_week'].isin(weekend_days)]
weekday_tracks = df[df['day_of_week'].isin(weekday_days)]

if len(weekend_tracks) > 1 and len(weekday_tracks) > 1:
    t_stat, p_value = stats.ttest_ind(weekend_tracks['valence'], weekday_tracks['valence'])
    print(f"\nT-test (выходные vs будни): p-value = {p_value:.4f}")
else:
    p_value = 1.0
    print("\nНедостаточно данных для статистического теста")
    
if difference > 0 and p_value < 0.05:
    print("Гипотеза подтверждается")
    print(f" в выходные значимо выше на {percent_difference:+.1f}%")
    print(f"Статистическая значимость: p = {p_value:.4f}")
    print("Люди действительно предпочитают более позитивную музыку в выходные")
    
else:
    print("Гипотеза не подтверждается")

Средняя валентность в будни:  0.551
Средняя валентность в выходные: 0.549
Разница: -0.001 (-0.3%)

T-test (выходные vs будни): p-value = 0.0000
Гипотеза не подтверждается


Гипотеза 5

In [ ]:
df['month'] = df['snapshot_date'].dt.month_name()
df['month_number'] = df['snapshot_date'].dt.month 

In [ ]:
df

,spotify_id,name,artists,daily_rank,daily_movement,weekly_movement,country,snapshot_date,popularity,is_explicit,...,liveness,valence,tempo,time_signature,tempo_category,tempo_range,day_of_week,day_number,month,month_number
50,2CGNAOSuO1MEFCbBRgUzjd,luther (with sza),"Kendrick Lamar, SZA",1,3,10,ZA,2025-02-17,90,False,...,0.2480,0.576,138.008,4,Быстрые,120-140,Monday,0,February,2
51,021iZJ1jjc6cZZ4sMXvcW9,Romeo & Juliet,"Naledi Aphiwe, Mawelele",2,3,2,ZA,2025-02-17,58,False,...,0.0593,0.691,125.989,3,Средние,120-140,Monday,0,February,2
52,24fRfn3FOM4xk02aE4wief,Vuka (feat. Thukuthela),"Oscar Mbo, Jazzworx, Thukuthela",3,-1,-2,ZA,2025-02-17,62,False,...,0.1050,0.524,118.006,4,Средние,100-120,Monday,0,February,2
53,2LeOHKRLm1Bz7eeNVinUYs,Esandleni,Nontokozo Mkhize,4,19,-2,ZA,2025-02-17,52,False,...,0.3350,0.495,99.768,4,Средние,80-100,Monday,0,February,2
54,4QdFYz1hjkDVDTqAKmIdOx,"Ngibolekeni (feat. Seun1401, LeeMcKrazy, Blxck...","DJ Maphorisa, Xduppy, Scotts Maphuma, Seun1401...",5,-4,-2,ZA,2025-02-17,59,False,...,0.0609,0.746,112.012,4,Средние,100-120,Monday,0,February,2
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1728377,0AYt6NMyyLd0rLuvr0UkMH,Slime You Out (feat. SZA),"Drake, SZA",46,4,0,AE,2023-10-18,84,True,...,0.2590,0.105,88.880,3,Средние,80-100,Wednesday,2,October,10
1728378,2Gk6fi0dqt91NKvlzGsmm7,SAY MY GRACE (feat. Travis Scott),"Offset, Travis Scott",47,3,0,AE,2023-10-18,80,True,...,0.1320,0.476,121.879,4,Средние,120-140,Wednesday,2,October,10
1728379,26b3oVLrRUaaybJulow9kz,People,Libianca,48,2,0,AE,2023-10-18,88,False,...,0.1020,0.693,124.357,5,Средние,120-140,Wednesday,2,October,10
1728380,5ydjxBSUIDn26MFzU3asP4,Rainy Days,V,49,1,0,AE,2023-10-18,88,False,...,0.1480,0.282,74.828,4,Средние,60-80,Wednesday,2,October,10


In [ ]:
def season_group(n):
    if n in [4, 5, 6, 7]:
        return 'Warm season'
    elif n in [10, 11, 12, 1]:
        return 'Cold season'
    else:
        return 'Other'

In [ ]:
df['season_group'] = df['month_number'].apply(season_group)
df

,spotify_id,name,artists,daily_rank,daily_movement,weekly_movement,country,snapshot_date,popularity,is_explicit,...,valence,tempo,time_signature,tempo_category,tempo_range,day_of_week,day_number,month,month_number,season_group
50,2CGNAOSuO1MEFCbBRgUzjd,luther (with sza),"Kendrick Lamar, SZA",1,3,10,ZA,2025-02-17,90,False,...,0.576,138.008,4,Быстрые,120-140,Monday,0,February,2,Other
51,021iZJ1jjc6cZZ4sMXvcW9,Romeo & Juliet,"Naledi Aphiwe, Mawelele",2,3,2,ZA,2025-02-17,58,False,...,0.691,125.989,3,Средние,120-140,Monday,0,February,2,Other
52,24fRfn3FOM4xk02aE4wief,Vuka (feat. Thukuthela),"Oscar Mbo, Jazzworx, Thukuthela",3,-1,-2,ZA,2025-02-17,62,False,...,0.524,118.006,4,Средние,100-120,Monday,0,February,2,Other
53,2LeOHKRLm1Bz7eeNVinUYs,Esandleni,Nontokozo Mkhize,4,19,-2,ZA,2025-02-17,52,False,...,0.495,99.768,4,Средние,80-100,Monday,0,February,2,Other
54,4QdFYz1hjkDVDTqAKmIdOx,"Ngibolekeni (feat. Seun1401, LeeMcKrazy, Blxck...","DJ Maphorisa, Xduppy, Scotts Maphuma, Seun1401...",5,-4,-2,ZA,2025-02-17,59,False,...,0.746,112.012,4,Средние,100-120,Monday,0,February,2,Other
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1728377,0AYt6NMyyLd0rLuvr0UkMH,Slime You Out (feat. SZA),"Drake, SZA",46,4,0,AE,2023-10-18,84,True,...,0.105,88.880,3,Средние,80-100,Wednesday,2,October,10,Cold season
1728378,2Gk6fi0dqt91NKvlzGsmm7,SAY MY GRACE (feat. Travis Scott),"Offset, Travis Scott",47,3,0,AE,2023-10-18,80,True,...,0.476,121.879,4,Средние,120-140,Wednesday,2,October,10,Cold season
1728379,26b3oVLrRUaaybJulow9kz,People,Libianca,48,2,0,AE,2023-10-18,88,False,...,0.693,124.357,5,Средние,120-140,Wednesday,2,October,10,Cold season
1728380,5ydjxBSUIDn26MFzU3asP4,Rainy Days,V,49,1,0,AE,2023-10-18,88,False,...,0.282,74.828,4,Средние,60-80,Wednesday,2,October,10,Cold season


In [ ]:
df_2 = df[df['season_group'].isin(['Warm season', 'Cold season'])]
audio_features = ['energy', 'danceability', 'valence', 'acousticness']

fig = make_subplots(
    rows=2, cols=2,
    subplot_titles=('Energy', 'Danceability', 'Valence', 'Acousticness'),
    vertical_spacing=0.1
)

for i, feature in enumerate(audio_features):
    row = i // 2 + 1
    col = i % 2 + 1
    
    fig.add_trace(go.Box(
        y=df_2[df_2['season_group'] == 'Warm season'][feature],
        name='Теплый сезон',
        marker_color='#FF6B6B',
        boxpoints='outliers'
    ), row=row, col=col)

    fig.add_trace(go.Box(
        y=df_2[df_2['season_group'] == 'Cold season'][feature],
        name='Холодный сезон',
        marker_color='#4ECDC4',
        boxpoints='outliers'
    ), row=row, col=col)

fig.update_layout(
    height=600,
    title_text="Сравнение аудиохарактеристик по сезонам релиза",
    showlegend=False
)
fig.show()


In [ ]:
for feature in audio_features:
    warm_data = df_2[df_2['season_group'] == 'Warm season'][feature]
    cold_data = df_2[df_2['season_group'] == 'Cold season'][feature]
    
    t_stat, p_value = stats.ttest_ind(warm_data, cold_data, equal_var=False)
    
    mean_diff = warm_data.mean() - cold_data.mean()
    
    print(f"\n{feature}:")
    print(f"  t = {t_stat:.3f}, p = {p_value:.4f}")
    print(f"  Разница: {mean_diff:.3f}")
    print(f"  Значимо: {'Да' if p_value < 0.05 else 'Нет'}")


energy:
  t = 41.903, p = 0.0000
  Разница: 0.013
  Значимо: Да

danceability:
  t = 78.424, p = 0.0000
  Разница: 0.020
  Значимо: Да

valence:
  t = 27.863, p = 0.0000
  Разница: 0.012
  Значимо: Да

acousticness:
  t = -68.732, p = 0.0000
  Разница: -0.032
  Значимо: Да


Гипотеза 6

In [ ]:
df['snapshot_year'] = pd.to_datetime(df['snapshot_date']).dt.year
df['release_year'] = pd.to_datetime(df['album_release_date']).dt.year

trend_data = df[df['release_year'].between(2018, 2025)] 

yearly_trends = trend_data.groupby('release_year')[['valence', 'energy', 'acousticness', 'danceability']].median().reset_index()

fig = px.line(yearly_trends, 
              x='release_year', 
              y=['valence', 'energy', 'acousticness'],
              title='Эволюция музыкальных трендов в топ-чартах Spotify (2018-2025)',
              labels={
                  'release_year': 'Год выхода альбома',
                  'value': 'Среднее значение показателя',
                  'variable': 'Аудиохарактеристики'
              },
              markers=True)

fig.update_layout(
    title_font_size=20,
    title_font_family="Arial",
    title_x=0.5,  
    width=1000,
    height=600,
    font=dict(size=12),
    legend=dict(
        orientation="h",
        yanchor="bottom",
        y=1.02,
        xanchor="right",
        x=1
    )
)

fig.for_each_trace(lambda t: t.update(name = {
    'valence': 'Valence (Позитивность)',
    'energy': 'Energy (Энергия)', 
    'acousticness': 'Acousticness (Акустичность)'
}[t.name]))

fig.update_xaxes(showgrid=True, gridwidth=1, gridcolor='LightGrey')
fig.update_yaxes(showgrid=True, gridwidth=1, gridcolor='LightGrey')

fig.show()

In [ ]:
from scipy.stats import linregress
trend_data = trend_data.groupby('release_year')[['valence', 'energy', 'acousticness']].mean().reset_index()

for feature in ['valence', 'energy', 'acousticness']:
    slope, intercept, r_value, p_value, std_err = linregress(trend_data['release_year'], trend_data[feature])
    print(f"Тренд для {feature}: Наклон = {slope:.4f}, R^2 = {r_value**2:.4f}")

Тренд для valence: Наклон = -0.0075, R^2 = 0.0956
Тренд для energy: Наклон = -0.0005, R^2 = 0.0006
Тренд для acousticness: Наклон = -0.0091, R^2 = 0.0945


In [ ]:
df['snapshot_date'].value_counts().reset_index()

,snapshot_date,count
0,2024-03-16,3888
1,2023-11-20,3811
2,2024-03-18,3781
3,2024-03-14,3717
4,2023-10-27,3714
...,...,...
472,2024-04-20,3100
473,2024-04-24,2500
474,2024-04-23,2100
475,2024-04-21,1900
